# **Nettoyage**

## Etapes
1. Imports et Chargement des données
2. Normalisation et dédoublonnage des numéros de TVA
3. Validation structurelle (algorithmes nationaux)
4. Construction de l'entonnoir chiffré
5. Estimation du temps d'appel VIES (benchmark sur échantillon)

---
## 1. Imports et Chargement des données

### Imports des bibliothèques

In [12]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

### Imports module `validation_structure.py`

In [ ]:
sys.path.append(str(Path("../api").resolve()))
from validation_structure import normaliser, valider, PAYS_COUVERTS

### Chargement des données

In [14]:
data_dir = Path("../data")
df = pd.read_csv(data_dir / "numeros_tva.csv", dtype = str, keep_default_na = False)

---
## 2. Normalisation et dédoublonnage des numéros de TVA

### Application de la fonction normaliser

In [15]:
df["tva_normalisee"] = df["numero_tva"].apply(normaliser)

### Détection des doublons après normalisation

In [16]:
marqueurs_vides = {"", "NA", "NULL"}
non_vide = ~df["tva_normalisee"].isin(marqueurs_vides)

frequences = df.loc[non_vide, "tva_normalisee"].value_counts()
numeros_doublons = frequences[frequences > 1]

display(numeros_doublons)

tva_normalisee
DK71704289        6
0282330178        5
DK91970813        4
NL785427788B39    4
PT100405711       4
                 ..
NL082237803B06    2
757548172         2
ZZ930766032       2
DK89608228        2
PL3605374689      2
Name: count, Length: 396, dtype: int64

### Sélectionner une seule ligne par numéro de TVA

In [17]:
df_avec_numero = df.loc[non_vide]
df_numeros_uniques = df_avec_numero.drop_duplicates("tva_normalisee")
df_sans_numero = df.loc[~non_vide]
df_dedoublonne = pd.concat([df_numeros_uniques, df_sans_numero])

display(df_dedoublonne.head())
print(df_dedoublonne.shape[0])

,id,raison_sociale,pays_declare,numero_tva,date_saisie,source_saisie,tva_normalisee
0,1,Nord Technologies GmbH,DK,DK73224645,2025-07-27,portail_client,DK73224645
1,2,Solaris Technologies SARL,PT,PT787327239,2024-03-29,reprise_erp,PT787327239
2,3,Verso Consulting NV,LU,LU71043234,2024-02-16,crm,LU71043234
3,4,Lumen Distribution Sp. z o.o.,SE,SE.751298146001,2024-02-27,import_fournisseur,SE751298146001
4,5,Textil Services Oy,NL,NL505862176B88,2025-04-14,crm,NL505862176B88


9562


### **Définition des doublons**

`normaliser()` du module `validation_structure.py` met le numéro en majuscules et retire les caractères non alphanumériques. Deux lignes sont des doublons si leur valeur `tva_normalisee` est identique. Les marqueurs vides sont exclus du comptage.

>**On considère comme doublons les lignes ayant le même numéro normalisé, hors valeurs vides. En gardant une seule ligne par numéro, on passe de 10 000 à 9 562 lignes, avant la validation structurelle.**

---
## 3. Validation structurelle

### Traitement des motifs du contrôle structurel des numéros de TVA

`valider()` contrôle la structure du numéro, pas son existence dans VIES.

| Motif | Traitement |
|---|---|
| `ok` | Numéro conforme au contrôle structurel ; candidat à une vérification VIES. |
| `vide` | Aucun numéro ; pas d'appel VIES. |
| `trop_court` | Numéro trop court ; pas d'appel VIES. |
| `cle_invalide` | Le numéro ne respecte pas le format national ou la clé de contrôle ; pas d'appel VIES. |
| `pays_non_couvert` | Le préfixe n'est pas répertorié dans `PAYS_COUVERTS` du module `validation_structure.py` ; garder à part, pas d'appel VIES. GB/UK est hors UE ; ZZ/QQ/XX ne correspondent à aucun pays. |

La colonne brute `numero_tva` est conservée dans `df`.

### Préparer et afficher la validation des numéros de TVA

In [18]:
tva_pour_validation = df_dedoublonne["tva_normalisee"].replace(
    {"NA": "", "NULL": ""}
)
resultats_validation = tva_pour_validation.apply(valider)

df_dedoublonne[["controle_structurel_ok", "motif_controle_structurel"]] = pd.DataFrame(
    resultats_validation.tolist(),
    columns=["controle_structurel_ok", "motif_controle_structurel"],
    index=df_dedoublonne.index,
)

display(df_dedoublonne[["tva_normalisee", "controle_structurel_ok", "motif_controle_structurel"]].head())
display(df_dedoublonne["motif_controle_structurel"].value_counts())

,tva_normalisee,controle_structurel_ok,motif_controle_structurel
0,DK73224645,False,cle_invalide
1,PT787327239,True,ok
2,LU71043234,True,ok
3,SE751298146001,True,ok
4,NL505862176B88,False,cle_invalide


motif_controle_structurel
ok                  5802
cle_invalide        2499
pays_non_couvert    1000
vide                 261
Name: count, dtype: int64

### Bilan du contrôle structurel

>Après avoir retiré les doublons, le tableau contient **9 562 lignes**.

Les résultats se répartissent selon les motifs suivants :

- **5 802 lignes : `ok`** (ces numéros sont candidats à une vérification par VIES, mais ne sont pas encore confirmés par ce service)
- **2 499 lignes : `cle_invalide`** (elles ne sont pas envoyées à VIES)
- **261 lignes : `vide`** (elles ne sont pas envoyées à VIES)
- **1 000 lignes : `pays_non_couvert`** (elles sont conservées à part, car le module ne permet pas de conclure sur leur validité)

---
## 4. Construction de l'entonnoir chiffré

### Filtrer les numéros candidats à VIES

In [25]:
df_candidats_vies = df_dedoublonne.loc[
    df_dedoublonne["motif_controle_structurel"] == "ok"
]
print(df_candidats_vies.shape[0])
df_candidats_vies.to_csv("../data/numeros_tva_nettoyes.csv", index=False)

5802


### Entonnoir chiffré

| Étape | Nombre de lignes |
|---|---:|
| Fichier de départ | 10 000 |
| Après avoir retiré les doublons | 9 562 |
| Candidats à vérifier dans VIES (`ok`) | 5 802 |

Le nettoyage évite **4 198 appels** par rapport à un appel pour chacune des 10 000 lignes initiales.

---
## 5. Estimation du temps d'appel VIES (benchmark sur échantillon)

### Sélectionner un échantillon à tester dans VIES

In [20]:
echantillon_vies = df_candidats_vies.groupby(
    df_candidats_vies["tva_normalisee"].str[:2]
).sample(n=2, random_state=42)

display(echantillon_vies[["tva_normalisee"]])

,tva_normalisee
1779,BE0903436719
8115,BE1766465119
3218,DK71641309
8008,DK31640253
2471,FI06746196
8827,FI57203730
841,FR07259968696
6303,FR08109418359
8778,IT20154311813
7737,IT64156663763


### Mesurer le temps de réponse de VIES

In [21]:
import time
import requests

mesures_vies = []

for numero_complet in echantillon_vies["tva_normalisee"]:
    pays = numero_complet[:2]
    numero = numero_complet[2:]
    url = f"https://ec.europa.eu/taxation_customs/vies/rest-api/ms/{pays}/vat/{numero}"

    debut = time.perf_counter()
    reponse = requests.get(url, timeout=30)
    temps = time.perf_counter() - debut
    reponse.raise_for_status()
    donnees = reponse.json()

    mesures_vies.append({
        "pays": pays,
        "numero_tva": numero_complet,
        "isValid": donnees.get("isValid"),
        "userError": donnees.get("userError"),
        "temps_reponse_s": temps,
    })

    time.sleep(1)

df_temps_vies = pd.DataFrame(mesures_vies)
display(df_temps_vies)

,pays,numero_tva,isValid,userError,temps_reponse_s
0,BE,BE0903436719,False,MS_MAX_CONCURRENT_REQ,0.259754
1,BE,BE1766465119,False,MS_MAX_CONCURRENT_REQ,0.251916
2,DK,DK71641309,False,INVALID,11.486799
3,DK,DK31640253,False,INVALID,14.551265
4,FI,FI06746196,False,INVALID,0.480321
5,FI,FI57203730,False,INVALID,0.419832
6,FR,FR07259968696,False,INVALID,7.947984
7,FR,FR08109418359,False,MS_MAX_CONCURRENT_REQ,0.128326
8,IT,IT20154311813,False,INVALID,0.303407
9,IT,IT64156663763,False,INVALID,0.255259


### Résumer les temps de réponse mesurés 

In [22]:
display(df_temps_vies["userError"].value_counts())

df_reponses_terminees = df_temps_vies.loc[
    df_temps_vies["userError"].isin(["VALID", "INVALID"])
]
display(df_reponses_terminees["temps_reponse_s"].describe())
display(
    df_reponses_terminees.groupby("pays")["temps_reponse_s"]
    .agg(["count", "mean", "median", "min", "max"])
)

userError
INVALID                  14
MS_MAX_CONCURRENT_REQ     5
VALID                     1
Name: count, dtype: int64

count    15.000000
mean      2.633237
std       4.678649
min       0.197468
25%       0.251055
50%       0.419832
75%       1.055165
max      14.551265
Name: temps_reponse_s, dtype: float64

,count,mean,median,min,max
pays,,,,,
DK,2,13.019032,13.019032,11.486799,14.551265
FI,2,0.450077,0.450077,0.419832,0.480321
FR,1,7.947984,7.947984,7.947984,7.947984
IT,2,0.279333,0.279333,0.255259,0.303407
LU,2,0.220421,0.220421,0.197468,0.243374
PL,2,0.228189,0.228189,0.209527,0.246850
PT,2,0.523067,0.523067,0.339348,0.706786
SE,2,1.055165,1.055165,1.037905,1.072424


### Tester 20 autres numéros de TVA au hasard

In [24]:
candidats_non_testes = df_candidats_vies.loc[
    ~df_candidats_vies.index.isin(echantillon_vies.index)
]

echantillon_vies_2 = candidats_non_testes.groupby(
    candidats_non_testes["tva_normalisee"].str[:2]
).sample(n=2, random_state=43)

mesures_vies_2 = []

for numero_complet in echantillon_vies_2["tva_normalisee"]:
    pays = numero_complet[:2]
    numero = numero_complet[2:]
    url = f"https://ec.europa.eu/taxation_customs/vies/rest-api/ms/{pays}/vat/{numero}"

    debut = time.perf_counter()
    reponse = requests.get(url, timeout=30)
    temps = time.perf_counter() - debut
    reponse.raise_for_status()
    donnees = reponse.json()

    mesures_vies_2.append({
        "pays": pays,
        "numero_tva": numero_complet,
        "isValid": donnees.get("isValid"),
        "userError": donnees.get("userError"),
        "temps_reponse_s": temps,
    })

    time.sleep(10)

df_temps_vies_2 = pd.DataFrame(mesures_vies_2)
display(df_temps_vies_2)
display(df_temps_vies_2["userError"].value_counts())

,pays,numero_tva,isValid,userError,temps_reponse_s
0,BE,BE0813672127,False,MS_MAX_CONCURRENT_REQ,0.136282
1,BE,BE1597858432,False,INVALID,5.559138
2,DK,DK72894545,False,MS_MAX_CONCURRENT_REQ,0.135554
3,DK,DK65470756,False,INVALID,14.710958
4,FI,FI32156376,True,VALID,0.321105
5,FI,FI80496922,False,INVALID,0.285302
6,FR,FR84788209317,False,MS_MAX_CONCURRENT_REQ,0.126026
7,FR,FR34641819584,False,MS_MAX_CONCURRENT_REQ,0.286630
8,IT,IT15638407518,False,INVALID,0.096756
9,IT,IT19567733514,False,INVALID,0.105324


userError
INVALID                  13
MS_MAX_CONCURRENT_REQ     6
VALID                     1
Name: count, dtype: int64